<a href="https://colab.research.google.com/github/VadimGolov/Collab_functions/blob/heicwebp_to_png/heicwebp_to_png.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
from IPython.display import display, clear_output, Javascript
import ipywidgets as widgets
import os, sys
import subprocess
import importlib
from pathlib import Path

def insert_function_code(f_path):
    # with open(f_path, 'r', encoding='utf-8') as f:
    #     code = f.read();

    js_code = f"""
        // var code = '{code}';
        var cell = Jupyter.notebook.insert_cell_at_bottom('code');
        // cell.set_text(code);
        // Jupyter.notebook.select_next();
        alert('Код успешно вставлен!');
    """

display(Javascript(js_code))


def get_git_branches():
    # Получаем список удаленных веток
    result = subprocess.run(
        ["git", "ls-remote", "--heads", "https://github.com/VadimGolov/Collab_functions.git"],
        capture_output=True, text=True
    )

    # Выбираем только имена веток
    branches = []
    for line in result.stdout.splitlines():
        branch = line.split()[-1].split('/')[-1]
        branches.append(branch)

    return branches

# Получаем список веток
available_branches = get_git_branches()


# Интерфейс
# branch_dropdown = widgets.Dropdown(
#     options=available_branches,
#     value='heicwebp_to_png',
#     description='🌿 Ветка/Функция:',
#     style={'description_width': 'initial'},
# )

# update_button = widgets.Button(
#     description='🔄 Загрузить из GitHub',
#     button_style='success',
#     icon='download'
# )

# output = widgets.Output()


def setup_environment(button=None):
    branch = branch_dropdown.value
    module_name = branch  # предполагаем, что имя файла и функции совпадает
    func_name = branch

    repo_path = Path('/content/Collab_functions')
    file_path = Path(repo_path, f'{func_name}.ipynb')

    with output:
        clear_output()
        print(f'🔄 Загружаем ветку {branch} и функцию {func_name}...')

        # Клонируем ветку
        subprocess.run(['rm', '-rf', 'Collab_functions'])
        subprocess.run([
            'git', 'clone', '--branch', branch,
            'https://github.com/VadimGolov/Collab_functions.git'
        ])

        if repo_path not in sys.path:
            sys.path.append(repo_path)

        try:
            # Автоперезагрузка
            %load_ext autoreload
            %autoreload 2

            # Импорт модуля и функции по имени ветки
            module = importlib.import_module(module_name)
            func = getattr(module, func_name)

            globals()[func_name] = func  # регистрируем глобально

            print(f'✅ Функция {func_name} успешно загружена!')

            # insert_function_code(file_path)
            print(file_path)
            insert_function_code(file_path)

        except Exception as e:
            print(f'⚠️ Ошибка при импорте: {e}')


# Интерфейс
branch_dropdown = widgets.Dropdown(
    options=available_branches,
    value='heicwebp_to_png',
    description='🌿 Ветка/Функция:',
    style={'description_width': 'initial'},
)

update_button = widgets.Button(
    description='🔄 Загрузить из GitHub',
    button_style='success',
    icon='download'
)

output = widgets.Output()

update_button.on_click(setup_environment)
display(widgets.VBox([branch_dropdown, update_button, output]))

In [104]:
from IPython.display import Javascript
from pathlib import Path
import google.colab.output

def insert_code_cell_from_file(file_path: Path):
    with open(file_path, 'r', encoding='utf-8') as f:
        code = f.read()

    # Экранируем кавычки и переносы
    js_safe_code = code.replace('\\', '\\\\').replace('`', '\\`')

    display(Javascript(f"""
        const code = `{js_safe_code}`;
        const cell = google.colab.kernel.invokeFunction(
            'notebook.insertCodeCell', [code], {{}}
        );
    """))


def _insert_code(code):
    from IPython.core.getipython import get_ipython
    ip = get_ipython()
    ip.set_next_input(code, replace=False)

google.colab.output.register_callback('notebook.insertCodeCell', _insert_code)

insert_code_cell_from_file('/content/Collab_functions/heicwebp_to_png.ipynb')

<IPython.core.display.Javascript object>